# Final submission

Trains the search-selected best model (`notebooks/modeling.ipynb`, `notes/accuracy_ceiling.md`: CatBoost binary classifier + fold-safe per-`ALLOCATION` target encoding, ~0.526 `TS`-grouped CV accuracy) on the full training set and writes a submission file.

## Setup

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import catboost as cb

import qrt_prep as P
import qrt_features as F

## Load, engineer, train

The allocation encoding is computed once from the *entire* training set (legitimate here, unlike inside CV, because every test-set row's `ALLOCATION` already appears in train -- 100% overlap -- and no test `TARGET` is used).

In [2]:
X_train, y_train, X_test = P.load_raw('../data/raw')
train_df = F.engineer(X_train.join(y_train))
test_df = F.engineer(X_test)

FEATURES = P.BASE_FEATURES + F.GROUP_DUMMY_COLS + F.MISSING_COLS + F.ROLLING_COLS
train_enc, test_enc = F.add_alloc_encoding(train_df, test_df, k=50)

y_sign = (train_df['target'] > 0).astype(int).to_numpy()
Xtr = np.column_stack([F.to_matrix(train_df, FEATURES), train_enc])
Xte = np.column_stack([F.to_matrix(test_df, FEATURES), test_enc])
Xtr.shape

(527073, 50)

In [3]:
model = cb.CatBoostClassifier(iterations=300, learning_rate=0.015, depth=6,
                               random_seed=42, verbose=False, thread_count=8)
model.fit(Xtr, y_sign)

CatBoostClassifier(depth=6, iterations=300, learning_rate=0.015, random_seed=42, thread_count=8, verbose=False)

## Predict and format

In [4]:
proba = model.predict_proba(Xte)[:, 1]
submission = pd.DataFrame({'prediction': (proba > 0.5).astype(int)}, index=test_df.index)
submission.index.name = 'ROW_ID'
submission['prediction'].value_counts(normalize=True)

prediction
1    0.589175
0    0.410825
Name: proportion, dtype: float64

## Sanity checks against sample_submission.csv

In [5]:
sample_submission = pd.read_csv('../data/raw/sample_submission.csv', index_col='ROW_ID')

assert submission.shape == sample_submission.shape
assert (submission.index == sample_submission.index).all()
assert set(submission['prediction'].unique()) <= {0, 1}
print('shape ok, index aligned, values in {0, 1}')
print('predicted positive share:', submission['prediction'].mean())
print('sample_submission positive share (random):', sample_submission['prediction'].mean())

shape ok, index aligned, values in {0, 1}
predicted positive share: 0.5891747725133354
sample_submission positive share (random): 0.5014747411358644


## Write

In [6]:
submission.to_csv('../submissions/catboost_alloc_encoding.csv')
pd.read_csv('../submissions/catboost_alloc_encoding.csv').head()

,ROW_ID,prediction
0,527073,0
1,527074,1
2,527075,0
3,527076,0
4,527077,0


## CV accuracy of this exact pipeline

Re-verifies the search's headline number (`notes/accuracy_ceiling.md`) using this notebook's own code, on the standard 5-fold `TS`-grouped CV.

In [7]:
folds = P.make_folds(train_df['TS'], n_splits=5, seed=0)
y = train_df['target'].to_numpy()
accs = []
for fold in range(5):
    val_mask = folds == fold
    tr, va = train_df[~val_mask], train_df[val_mask]
    tr_enc, va_enc = F.add_alloc_encoding(tr, va, k=50)
    Xtr_f = np.column_stack([F.to_matrix(tr, FEATURES), tr_enc])
    Xva_f = np.column_stack([F.to_matrix(va, FEATURES), va_enc])
    fold_model = cb.CatBoostClassifier(iterations=300, learning_rate=0.015, depth=6,
                                        random_seed=42, verbose=False, thread_count=8)
    fold_model.fit(Xtr_f, (y[~val_mask] > 0).astype(int))
    pred = fold_model.predict_proba(Xva_f)[:, 1]
    accs.append(((pred > 0.5).astype(int) == (y[val_mask] > 0).astype(int)).mean())

print('mean CV accuracy:', np.mean(accs), accs)

mean CV accuracy: 0.5256856877085102 [np.float64(0.5256116137794337), np.float64(0.5255303045005532), np.float64(0.5272668510755602), np.float64(0.524917540095833), np.float64(0.5251021290911713)]


This CatBoost + allocation-encoding pipeline reaches ~0.526 mean CV accuracy -- the best validated result from an extensive search (150+ configurations across feature engineering, 3 GBM families, hierarchical encoding, ensembling, and neural sequence models; see `notes/accuracy_ceiling.md`). It comfortably beats the published benchmark's 0.5079 public score, but the search found this to be close to a real ceiling for this feature set, not a starting point for much further improvement.